# Day 9 v1 — Silver Transformation: Charging Sessions (Blob CSV → Silver Delta)

**Source:** `/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/realtime/charging_sessions/`
**Sink:** `/Volumes/dbw_ev_intelligence_dev/default/silver-volume/realtime/charging_sessions/` (Delta)

This is the **learning version** — one entity (charging_sessions), every step written explicitly.
Same teaching pattern as Day 8 v1 but for Blob CSV data instead of API JSON.

**Blob CSV Bronze structure:**
```
bronze-volume/realtime/charging_sessions/YYYY/MM/DD/HH/sessions_YYYYMMDD_HHMM.csv
```

**Steps:**
1. Read all Bronze CSV files for charging_sessions
2. Cast columns to correct types
3. Trim whitespace from string columns
4. Add Silver audit columns
5. Deduplicate on `session_id`
6. Write to Silver as Delta table (full overwrite)
7. Verify the output

In [0]:
# Cell 1 — Imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timezone

print("Imports OK")

Imports OK


In [0]:
display(dbutils.fs.ls("/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/realtime/charging_sessions/2026/06/01/06"))

path,name,size,modificationTime
dbfs:/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/realtime/charging_sessions/2026/06/01/06/sessions_20260601_0600.csv,sessions_20260601_0600.csv,10773225,1783745990000


In [0]:
# Cell 2 — Constants
BRONZE_PATH = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/realtime/charging_sessions"
SILVER_PATH = "/Volumes/dbw_ev_intelligence_dev/default/silver-volume/realtime/charging_sessions"

RUN_TS = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

print(f"Bronze path : {BRONZE_PATH}")
print(f"Silver path : {SILVER_PATH}")
print(f"Run time    : {RUN_TS}")

Bronze path : /Volumes/dbw_ev_intelligence_dev/default/bronze-volume/realtime/charging_sessions
Silver path : /Volumes/dbw_ev_intelligence_dev/default/silver-volume/realtime/charging_sessions
Run time    : 2026-07-15T12:43:23Z


In [ ]:
# Cell 3 — Read all Bronze CSV files
# Unity Catalog requires _metadata.file_path (input_file_name() is blocked).
# IMPORTANT: Do NOT use .select("*", "_metadata.file_path") — the wildcard "*"
# combined with a metadata column causes column count mismatch and shifts values.
# Instead, list every CSV column explicitly alongside _metadata.file_path.
from pyspark.sql.functions import col, regexp_extract, concat_ws, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType

SCHEMA = StructType([
    StructField("session_id",     StringType(), True),
    StructField("charger_id",     StringType(), True),
    StructField("vehicle_id",     StringType(), True),
    StructField("station_id",     StringType(), True),
    StructField("customer_id",    StringType(), True),
    StructField("plug_in_ts",     StringType(), True),
    StructField("charge_end_ts",  StringType(), True),
    StructField("energy_kwh",     StringType(), True),
    StructField("session_status", StringType(), True),
    StructField("tariff_id",      StringType(), True),
])

CSV_COLS = [f.name for f in SCHEMA.fields]

raw_df = (
    spark.read
    .option("header", "true")
    .option("recursiveFileLookup", "true")
    .schema(SCHEMA)
    .csv(BRONZE_PATH)
    .select(
        *[col(c) for c in CSV_COLS],
        col("_metadata.file_path").alias("_file_path")
    )
)

raw_df = (
    raw_df
    .withColumn("_year",  regexp_extract(col("_file_path"), r"/(\d{4})/", 1))
    .withColumn("_month", regexp_extract(col("_file_path"), r"/\d{4}/(\d{2})/", 1))
    .withColumn("_day",   regexp_extract(col("_file_path"), r"/\d{4}/\d{2}/(\d{2})/", 1))
    .withColumn("_hour",  regexp_extract(col("_file_path"), r"/\d{4}/\d{2}/\d{2}/(\d{2})/", 1))
    .withColumn(
        "updated_at",
        to_timestamp(
            concat_ws(" ",
                concat_ws("-", col("_year"), col("_month"), col("_day")),
                col("_hour")
            ),
            "yyyy-MM-dd HH"
        )
    )
    .drop("_file_path", "_year", "_month", "_day", "_hour")
)

print(f"Row count: {raw_df.count()}")
raw_df.show(3, truncate=True)

In [0]:
# Cell 4 — Cast columns to correct types
# inferSchema may guess wrong (e.g. timestamps as strings).
# Explicit casting guarantees consistent Silver schema across all runs.

typed_df = raw_df.select(
    F.col("session_id").cast("string"),
    F.col("vehicle_id").cast("string"),
    F.col("charger_id").cast("string"),
    F.col("plug_in_ts").cast("timestamp"),
    F.col("charge_end_ts").cast("timestamp"),
    F.col("energy_kwh").cast("decimal(10,4)"),
    F.col("session_status").cast("string"),
    F.col("tariff_id").cast("string"),
    F.col("updated_at").cast("timestamp")

)

print("After type casting:")
typed_df.printSchema()

After type casting:
root
 |-- session_id: string (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- charger_id: string (nullable = true)
 |-- plug_in_ts: timestamp (nullable = true)
 |-- charge_end_ts: timestamp (nullable = true)
 |-- energy_kwh: decimal(10,4) (nullable = true)
 |-- session_status: string (nullable = true)
 |-- tariff_id: string (nullable = true)
 |-- updated_at: timestamp (nullable = false)



root
 |-- session_id: string (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- charger_id: string (nullable = true)
 |-- plug_in_ts: timestamp (nullable = true)
 |-- charge_end_ts: timestamp (nullable = true)
 |-- energy_kwh: decimal(10,4) (nullable = true)
 |-- session_status: string (nullable = true)
 |-- tariff_id: string (nullable = true)
 |-- updated_at: timestamp (nullable = false)
 |-- silver_ingested_at: timestamp (nullable = true)
 |-- silver_load_type: string (nullable = false)
 |-- silver_pipeline: string (nullable = false)



In [0]:
# Cell 5 — Trim whitespace from all string columns
# CSV files can have trailing spaces in string fields.

string_cols = [c for c, t in typed_df.dtypes if t == "string"]
trimmed_df  = typed_df
for col in string_cols:
    trimmed_df = trimmed_df.withColumn(col, F.trim(F.col(col)))

print(f"Trimmed string columns: {string_cols}")

Trimmed string columns: ['session_id', 'vehicle_id', 'charger_id', 'session_status', 'tariff_id']


+---------------+------------------+------------+---------------+---------------+-------------------+-------------------+---------+-------------------------------------------------------------------------------------------------------------------------------+----+-----+---+----+-------------------+--------------+
|session_id     |charger_id        |vehicle_id  |plug_in_ts     |charge_end_ts  |energy_kwh         |session_status     |tariff_id|file_path                                                                                                                      |year|month|day|hour|updated_at         |invalid_column|
+---------------+------------------+------------+---------------+---------------+-------------------+-------------------+---------+-------------------------------------------------------------------------------------------------------------------------------+----+-----+---+----+-------------------+--------------+
|SESS-0000000001|CHR-EVL-DC-0048717|STN-AU-00422|VEH-TS

In [0]:
# Cell 6 — Add Silver audit columns
# Same pattern as Day 8 — every Silver row gets lineage columns.

audited_df = (
    trimmed_df
    .withColumn("silver_ingested_at", F.lit(RUN_TS).cast("timestamp"))
    .withColumn("silver_load_type",   F.lit("full"))
    .withColumn("silver_pipeline",    F.lit("pl_silver_blob_charging_sessions_v1"))
)

print("After adding audit columns:")
audited_df.printSchema()

After adding audit columns:
root
 |-- session_id: string (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- charger_id: string (nullable = true)
 |-- plug_in_ts: timestamp (nullable = true)
 |-- charge_end_ts: timestamp (nullable = true)
 |-- energy_kwh: decimal(10,4) (nullable = true)
 |-- session_status: string (nullable = true)
 |-- tariff_id: string (nullable = true)
 |-- updated_at: timestamp (nullable = false)
 |-- silver_ingested_at: timestamp (nullable = true)
 |-- silver_load_type: string (nullable = false)
 |-- silver_pipeline: string (nullable = false)



In [0]:
# Cell 7 — Deduplicate on session_id
# Bronze may contain the same session_id across multiple hourly CSV files
# (e.g. an in-progress session updated across two hours).
# Keep the row with the most recent updated_at per session_id.

window = Window.partitionBy("session_id").orderBy(F.col("updated_at").desc())

deduped_df = (
    audited_df
    .withColumn("_row_num", F.row_number().over(window))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)

before = audited_df.count()
after  = deduped_df.count()
print(f"Before dedup : {before}")
print(f"After dedup  : {after}")
print(f"Duplicates removed : {before - after}")

Before dedup : 3010612
After dedup  : 1000000
Duplicates removed : 2010612


In [0]:
deduped_df.printSchema()

root
 |-- session_id: string (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- charger_id: string (nullable = true)
 |-- plug_in_ts: timestamp (nullable = true)
 |-- charge_end_ts: timestamp (nullable = true)
 |-- energy_kwh: decimal(10,4) (nullable = true)
 |-- session_status: string (nullable = true)
 |-- tariff_id: string (nullable = true)
 |-- updated_at: timestamp (nullable = false)
 |-- silver_ingested_at: timestamp (nullable = true)
 |-- silver_load_type: string (nullable = false)
 |-- silver_pipeline: string (nullable = false)



In [0]:
display(raw_df.limit(5))

session_id,charger_id,vehicle_id,plug_in_ts,charge_end_ts,energy_kwh,session_status,tariff_id,file_path,year,month,day,hour,updated_at
SESS-0000000001,CHR-EVL-DC-0000002,VEH-HYN-0000002,2023-11-21T02:39:49,2023-11-21T03:29:49,13.85,Cancelled,TAR-AU-00002,dbfs:/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/realtime/charging_sessions/2026/06/19/06/charging_sessions_20260619_0600.csv,2026,06,19,06,2026-06-19T06:00:00.000Z
SESS-0000000002,CHR-TRI-DC-0000003,VEH-KIA-0000003,2025-11-24T19:29:01,2025-11-24T20:27:01,32.07,Completed,TAR-AU-00003,dbfs:/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/realtime/charging_sessions/2026/06/19/06/charging_sessions_20260619_0600.csv,2026,06,19,06,2026-06-19T06:00:00.000Z
SESS-0000000003,CHR-SIE-DC-0000004,VEH-BYD-0000004,2025-11-03T19:13:00,2025-11-03T20:56:00,57.81,Failed,TAR-AU-00004,dbfs:/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/realtime/charging_sessions/2026/06/19/06/charging_sessions_20260619_0600.csv,2026,06,19,06,2026-06-19T06:00:00.000Z
SESS-0000000004,CHR-WAL-DC-0000005,VEH-NIS-0000005,2026-02-02T16:33:41,2026-02-02T17:33:41,30.86,Cancelled,TAR-AU-00005,dbfs:/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/realtime/charging_sessions/2026/06/19/06/charging_sessions_20260619_0600.csv,2026,06,19,06,2026-06-19T06:00:00.000Z
SESS-0000000005,CHR-SCH-DC-0000006,VEH-MER-0000006,2023-09-06T18:14:44,2023-09-06T18:55:44,115.51,Completed,TAR-AU-00006,dbfs:/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/realtime/charging_sessions/2026/06/19/06/charging_sessions_20260619_0600.csv,2026,06,19,06,2026-06-19T06:00:00.000Z


In [0]:
deduped_df.cache()
deduped_df.count()

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6379816847569138>, line 1
----> 1 deduped_df.cache()
      2 deduped_df.count()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:2133, in DataFrame.cache(self)
   2132 def cache(self) -> ParentDataFrame:
-> 2133     return self.persist()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:2140, in DataFrame.persist(self, storageLevel)
   2135 def persist(
   2136     self,
   2137     storageLevel: StorageLevel = (StorageLevel.MEMORY_AND_DISK_DESER),
   2138 ) -> ParentDataFrame:
   2139     relation = self._plan.plan(self._session.client)
-> 2140     self._session.client._analyze(
   2141         method="persist", relation=relation, storage_level=storageLevel
   2142     )
   2143     return self

File /databricks/python/lib/python3.12/site-packages/

In [0]:
# Cell 8 — Write to Silver as Delta table (full overwrite)

(
    deduped_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_PATH)
)

print(f"Written to: {SILVER_PATH}")
print(f"Rows written: {deduped_df.count()}")

---------------------------------------------------------------------------
DateTimeException                         Traceback (most recent call last)
File <command-6379816847569130>, line 8
      1 # Cell 8 — Write to Silver as Delta table (full overwrite)
      3 (
      4     deduped_df.write
      5     .format("delta")
      6     .mode("overwrite")
      7     .option("overwriteSchema", "true")
----> 8     .save(SILVER_PATH)
      9 )
     11 print(f"Written to: {SILVER_PATH}")
     12 print(f"Rows written: {deduped_df.count()}")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:703, in DataFrameWriter.save(self, path, format, mode, partitionBy, **options)
    701     self.format(format)
    702 self._write.path = path
--> 703 _, _, ei = self._spark.client.execute_command(
    704     self._write.command(self._spark.client), self._write.observations
    705 )
    706 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/py

In [0]:
# Cell 9 — Verify Silver output

silver_df = spark.read.format("delta").load(SILVER_PATH)

print("Silver charging_sessions schema:")
silver_df.printSchema()
print(f"\nTotal rows: {silver_df.count()}")
silver_df.show(5, truncate=True)

print("\nNull check on session_id (should be 0):")
print(silver_df.filter(F.col("session_id").isNull()).count())

print("\nStatus distribution:")
silver_df.groupBy("status").count().orderBy("count", ascending=False).show()